In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import zipfile
import random
import shutil

# JALUR PATH SESUAI GAMBAR LU (Gdrive -> Semester 8 -> Computer Vision)
ZIP_PATH = '/content/drive/MyDrive/Semester 8/Computer Vision/dataset_beton.zip'

# 1. Proses Ekstrak file ZIP dari GDrive ke local session Colab
print("Sedang mengekstrak dataset dari GDrive, tunggu bentar...")
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall('raw_dataset')

# 2. Bikin folder baru buat latihan (Cuma ambil 1000 gambar biar cepet)
base_dir = 'dataset_latihan'
os.makedirs(os.path.join(base_dir, 'cracked'), exist_ok=True)
os.makedirs(os.path.join(base_dir, 'normal'), exist_ok=True)

# 3. Fungsi buat mindahin 1000 gambar secara acak
def ambil_1000_gambar(source_folder, target_folder):
    semua_gambar = os.listdir(source_folder)
    gambar_terpilih = random.sample(semua_gambar, 1000)

    for nama_file in gambar_terpilih:
        path_asal = os.path.join(source_folder, nama_file)
        path_tujuan = os.path.join(target_folder, nama_file)
        shutil.copy(path_asal, path_tujuan)

# 4. Eksekusi pemindahan (Pastikan nama folder di dalam ZIP lu adalah 'Positive' dan 'Negative')
ambil_1000_gambar('raw_dataset/Positive', 'dataset_latihan/cracked')
ambil_1000_gambar('raw_dataset/Negative', 'dataset_latihan/normal')

print("Selesai! Folder 'dataset_latihan' udah siap dengan masing-masing 1000 gambar.")

Sedang mengekstrak dataset dari GDrive, tunggu bentar...
Selesai! Folder 'dataset_latihan' udah siap dengan masing-masing 1000 gambar.


In [3]:
import tensorflow as tf
from tensorflow.keras import layers, models

# 1. Tentukan ukuran gambar dan batch size
# Kita samain ukurannya jadi 128x128 pixel biar enteng tapi detail retaknya tetep kelihatan
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

# 2. Load data untuk TRAINING (80% dari total data)
train_dataset = tf.keras.utils.image_dataset_from_directory(
    'dataset_latihan',
    validation_split=0.2,       # 20% disimpan buat ujian/validasi
    subset="training",
    seed=123,                   # Angka acak biar pembagiannya konsisten
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# 3. Load data untuk VALIDASI/UJIAN (20% sisanya)
val_dataset = tf.keras.utils.image_dataset_from_directory(
    'dataset_latihan',
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# 4. Cek label yang dideteksi oleh Python
class_names = train_dataset.class_names
print("Label yang terdeteksi:", class_names)

Found 2000 files belonging to 2 classes.
Using 1600 files for training.
Found 2000 files belonging to 2 classes.
Using 400 files for validation.
Label yang terdeteksi: ['cracked', 'normal']


In [4]:
# 1. Bikin Arsitektur Model CNN
model = models.Sequential([
    # Rescaling: Mengubah nilai piksel dari 0-255 jadi 0-1 biar CNN belajarnya lebih stabil
    layers.Rescaling(1./255, input_shape=(128, 128, 3)),

    # Layer Ring 1: Nyari pola garis makro
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # Layer Ring 2: Nyari pola retakan yang lebih detail
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # Layer Ring 3: Makin dalam biar makin pinter
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # Flatten: Mengubah gambar 2D jadi 1D sebelum masuk ke tebakan final
    layers.Flatten(),
    layers.Dense(128, activation='relu'),

    # Output Layer: Cuma 1 neuron karena klasifikasi biner (Retak atau Normal)
    layers.Dense(1, activation='sigmoid')
])

# 2. Compile Model (Ngasih tahu cara koreksi jawaban salah)
model.compile(
    optimizer='adam',
    loss='binary_crossentropy', # Khusus buat 2 kelas
    metrics=['accuracy']
)

# Tampilkan struktur model lu di layar
model.summary()

# 3. Mulai Training (AI Mulai Belajar Selama 10 Putaran/Epochs)
print("\n--- AI Mulai Belajar ---")
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=20
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/preprocessing/data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     3,211,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,304,769 (12.61 MB)

 Trainable params: 3,304,769 (12.61 MB)

 Non-trainable params: 0 (0.00 B)


--- AI Mulai Belajar ---
Epoch 1/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 9s 47ms/step - accuracy: 0.6925 - loss: 0.5813 - val_accuracy: 0.9600 - val_loss: 0.1642
Epoch 2/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.9606 - loss: 0.1508 - val_accuracy: 0.9725 - val_loss: 0.0813
Epoch 3/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.9650 - loss: 0.1230 - val_accuracy: 0.9775 - val_loss: 0.0795
Epoch 4/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.9706 - loss: 0.1109 - val_accuracy: 0.9700 - val_loss: 0.1066
Epoch 5/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.8831 - loss: 0.3322 - val_accuracy: 0.9250 - val_loss: 0.3078
Epoch 6/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.9638 - loss: 0.1334 - val_accuracy: 0.9800 - val_loss: 0.0841
Epoch 7/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - accuracy: 0.9806 - loss: 0.0806 - val_accuracy: 0.9800 - val_loss: 0.0798
Epoch 8/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 28ms/step - accuracy: 0.9856 - loss: 0.0624 

In [5]:
import numpy as np
from google.colab import files
from tensorflow.keras.preprocessing import image

print("Silahkan upload foto dinding bebas (.jpg/.png) buat dites")
uploaded = files.upload()

for fn in uploaded.keys():
    # 1. Load gambar yang di-upload dan sesuaikan ukurannya ke 128x128
    path = fn
    img = image.load_img(path, target_size=(128, 128))

    # 2. Ubah gambar jadi array angka buat dibaca CNN
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0) # Nambahin dimensi batch

    # 3. AI mulai nebak
    images = np.vstack([x])
    prediction = model.predict(images)

    # 4. Tampilkan Hasil (Menggunakan logika Sigmoid: dekat ke 0 = Cracked, dekat ke 1 = Normal)
    # Catatan: Urutan label sesuai alphabet folder, biasanya 'cracked' itu indeks 0, 'normal' indeks 1.
    score = prediction[0][0]

    print("\n=== HASIL ANALISIS AI ===")
    if score < 0.5:
        confidence = (1 - score) * 100
        print(f"🔴 STATUS: RETAK (CRACKED)")
        print(f"🎯 Keyakinan AI: {confidence:.2f}%")
    else:
        confidence = score * 100
        print(f"🟢 STATUS: NORMAL (AMAN)")
        print(f"🎯 Keyakinan AI: {confidence:.2f}%")
    print("=========================\n")

Silahkan upload foto dinding bebas (.jpg/.png) buat dites


Saving 19993.jpg to 19993.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 649ms/step

=== HASIL ANALISIS AI ===
🔴 STATUS: RETAK (CRACKED)
🎯 Keyakinan AI: 100.00%



In [6]:
model.save('model_cracked_normal.h5')
print("Model berhasil disimpan dengan nama 'model_cracked_normal.h5'")

Model berhasil disimpan dengan nama 'model_cracked_normal.h5'
